# quantum2circuit — research scaffold (Google Colab)

This notebook builds a modular `quantum2circuit/` repository and runs a verified one-dimensional nearest-neighbour NEGF demonstration. It is a scaffold for the DFT → Wannier/TB → NEGF → noise/compact-model → SPICE programme, not a complete production implementation of DFT, self-consistent Poisson–NEGF, or atomistic spin/valley transport.

**Suggested sequence:** run all cells; inspect generated modules; add validated DFT/Wannier inputs; compare against experimental data; only then extend beyond the toy chain. The numerical chain is intended to verify software plumbing and physics invariants.


In [ ]:
from pathlib import Path
import sys, json, textwrap, subprocess, os
ROOT = Path('/content/quantum2circuit')
for d in ['configs','materials','dft','wannier','tightbinding','negf','electrostatics','noise','spinvalley','electrothermal','compact','spice','experiments','analysis','notebooks','tests']:
    (ROOT/d).mkdir(parents=True, exist_ok=True)
print('Repository root:', ROOT)


## 1. Write the modular starter source files

In [ ]:
FILES = {'README.md': '# quantum2circuit\n\nA modular research scaffold for a traceable materials-to-circuit workflow.\n\n## Scope\n- Working baseline: 1D nearest-neighbour chain, analytic semi-infinite lead self-energy, coherent NEGF transmission and Landauer current.\n- Starter utilities: noise PSD models, thermal RC state, compact-model fitting, CSV experiment import, SPICE netlist export.\n- Planned adapters: ASE + Quantum ESPRESSO/GPAW; Wannier90; Kwant or custom sparse NEGF; ngspice. These need validation for each material/device.\n\n## Scientific safeguards\n1. Preserve raw data and metadata.\n2. Record units, code versions, input hashes, convergence settings and random seeds.\n3. Never describe the toy chain as a realistic material prediction.\n4. Separate fitted, measured and assumed parameters.\n5. Use held-out conditions and uncertainty intervals.\n', 'pyproject.toml': '[project]\nname = "quantum2circuit"\nversion = "0.1.0"\ndescription = "Research scaffold from quantum transport to compact circuit models"\nrequires-python = ">=3.10"\ndependencies = ["numpy", "scipy", "pandas", "matplotlib"]\n', 'negf/chain.py': '"""Transparent coherent NEGF for a 1D uniform chain with identical semi-infinite leads.\nEnergies use eV; hbar and q are not needed for transmission. The current helper returns\ncurrent in amperes when energies are in joules, so use landauer_current_ev for eV grids.\n"""\nimport numpy as np\n\ndef surface_gf_1d(E, onsite, hopping):\n    """Retarded surface Green function of a semi-infinite nearest-neighbour chain.\n    hopping is real and nonzero; returns the retarded branch.\n    """\n    z = np.asarray(E, dtype=complex) - onsite\n    t = float(hopping)\n    root = np.sqrt(z*z - 4*t*t + 0j)\n    # Select branch yielding Im(g_surface) <= 0 for retarded Green function.\n    g1 = (z - root)/(2*t*t)\n    g2 = (z + root)/(2*t*t)\n    g = np.where(np.imag(g1) <= 1e-12, g1, g2)\n    return g\n\ndef transmission_chain(E, onsite, hopping, n_sites, contact_hopping=None, eta=1e-9):\n    """Transmission for an N-site chain connected to identical 1D leads.\n    This dense implementation is for tests/tutorials, not large devices.\n    """\n    if n_sites < 1: raise ValueError(\'n_sites must be >= 1\')\n    t = float(hopping); tc = t if contact_hopping is None else float(contact_hopping)\n    Earr = np.atleast_1d(E).astype(float); out=[]\n    for en in Earr:\n        gs = surface_gf_1d(en, onsite, t)\n        sig = tc*tc*gs\n        gamma = max(0.0, -2*np.imag(sig))\n        H = np.diag(np.full(n_sites, onsite, dtype=complex))\n        if n_sites > 1:\n            H += np.diag(np.full(n_sites-1, t), 1) + np.diag(np.full(n_sites-1, t), -1)\n        M = (en+1j*eta)*np.eye(n_sites) - H\n        M[0,0] -= sig; M[-1,-1] -= sig\n        G = np.linalg.inv(M)\n        out.append(float(np.real(gamma*gamma*abs(G[0,-1])**2)))\n    return np.array(out) if np.ndim(E) else out[0]\n\ndef fermi(E, mu, kT):\n    """Stable Fermi occupation for E, mu and kT in eV."""\n    x=np.clip((np.asarray(E)-mu)/kT, -700, 700)\n    return 1.0/(1.0+np.exp(x))\n\ndef landauer_current_ev(E, T, mu_s, mu_d, temperature_K):\n    """Current (A) using I=(2q^2/h) integral T(E)[fS-fD] dE_J.\n    E and chemical potentials are eV; factor 2 assumes spin degeneracy.\n    """\n    q=1.602176634e-19; h=6.62607015e-34; kB_eV=8.617333262e-5\n    E=np.asarray(E,float); T=np.asarray(T,float); kT=kB_eV*temperature_K\n    fs=fermi(E,mu_s,kT); fd=fermi(E,mu_d,kT)\n    integral_eV=np.trapezoid(T*(fs-fd), E)\n    return (2*q*q/h)*integral_eV  # eV integration converts by q; one q from Landauer charge\n', 'noise/psd.py': '"""Starter noise models. PSD conventions must be matched to instrument/data conventions."""\nimport numpy as np\ndef one_over_f(f, amplitude, exponent=1.0, white=0.0):\n    f=np.asarray(f,float)\n    if np.any(f<=0): raise ValueError(\'frequency must be positive\')\n    return white + amplitude/(f**exponent)\ndef lorentzian_rts(f, delta_i, tau):\n    f=np.asarray(f,float)\n    if tau<=0: raise ValueError(\'tau must be positive\')\n    return 4*(delta_i**2)*tau/(1+(2*np.pi*f*tau)**2)\ndef fit_noise_components(f, psd):\n    """Placeholder for weighted nonlinear fit; returns a log-log diagnostic only."""\n    f=np.asarray(f,float); psd=np.asarray(psd,float)\n    return {\'n_points\':int(len(f)), \'frequency_min_Hz\':float(np.min(f)), \'frequency_max_Hz\':float(np.max(f)),\n            \'psd_min\':float(np.min(psd)), \'psd_max\':float(np.max(psd)),\n            \'warning\':\'No trap count or mechanism should be inferred from this summary alone.\'}\n', 'electrothermal/rc.py': '"""Lumped electrothermal starter: Cth dT/dt = P - (T-Tamb)/Rth."""\nimport numpy as np\ndef simulate_temperature(time_s, power_W, Tamb_K, Rth_K_per_W, Cth_J_per_K, T0_K=None):\n    t=np.asarray(time_s,float); p=np.asarray(power_W,float)\n    if len(t)!=len(p) or len(t)<2: raise ValueError(\'time and power arrays must match and contain >=2 points\')\n    if np.any(np.diff(t)<=0) or Rth_K_per_W<=0 or Cth_J_per_K<=0: raise ValueError(\'invalid time or thermal parameters\')\n    T=np.empty_like(t); T[0]=Tamb_K if T0_K is None else T0_K\n    for i,dt in enumerate(np.diff(t),start=1):\n        dTdt=(p[i-1]-(T[i-1]-Tamb_K)/Rth_K_per_W)/Cth_J_per_K\n        T[i]=T[i-1]+dt*dTdt\n    return T\n', 'compact/models.py': '"""Simple, explicitly phenomenological compact-model starter."""\nimport numpy as np\ndef softplus(x): return np.logaddexp(0.0, x)\ndef toy_fet_current(vg, vd, vth=0.4, beta=1e-4, slope=0.08):\n    """Toy continuous current law for software demos only; not a foundry model."""\n    vg=np.asarray(vg,float); vd=np.asarray(vd,float)\n    overdrive=slope*softplus((vg-vth)/slope)\n    return beta*overdrive*vd/(1+np.abs(vd))\n', 'experiments/import_csv.py': '"""Import measurement CSV without overwriting raw files."""\nimport pandas as pd\ndef load_measurements(path, required_columns=None):\n    df=pd.read_csv(path)\n    if required_columns:\n        missing=set(required_columns)-set(df.columns)\n        if missing: raise ValueError(f\'Missing required columns: {sorted(missing)}\')\n    return df\n', 'spice/netlist.py': '"""Minimal text netlist helper; validate generated circuits with installed ngspice."""\ndef dc_resistor_divider(vin=1.0, r1=1000.0, r2=1000.0):\n    if min(r1,r2)<=0: raise ValueError(\'resistances must be positive\')\n    return f"* generated by quantum2circuit\\nV1 in 0 DC {vin}\\nR1 in out {r1}\\nR2 out 0 {r2}\\n.op\\n.end\\n"\n', 'analysis/provenance.py': '"""Small provenance manifest helper."""\nimport hashlib, json, platform, datetime\nfrom pathlib import Path\ndef sha256_file(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1<<20),b\'\'): h.update(b)\n    return h.hexdigest()\ndef write_manifest(output, inputs, parameters):\n    manifest={\'created_utc\':datetime.datetime.now(datetime.timezone.utc).isoformat(),\n              \'python\':platform.python_version(),\'inputs\':{str(p):sha256_file(p) for p in inputs},\'parameters\':parameters}\n    Path(output).write_text(json.dumps(manifest,indent=2)); return manifest\n'}
for rel, content in FILES.items():
    p=ROOT/rel; p.parent.mkdir(parents=True,exist_ok=True); p.write_text(content)
for d in ['materials','dft','wannier','tightbinding','electrostatics','spinvalley','tests']:
    p=ROOT/d/'__init__.py'
    if not p.exists(): p.write_text('"""quantum2circuit module."""\n')
print('Wrote', len(FILES), 'source files under', ROOT)


## 2. Run the NEGF demo

In [ ]:
import numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, str(ROOT))
from negf.chain import transmission_chain, landauer_current_ev
E=np.linspace(-3.0,3.0,601)
T=transmission_chain(E, onsite=0.0, hopping=-1.0, n_sites=12)
plt.figure(figsize=(8,4)); plt.plot(E,T); plt.xlabel('Energy (eV)'); plt.ylabel('Transmission'); plt.title('Toy 1D chain: NEGF transmission'); plt.grid(True); plt.show()
I=landauer_current_ev(E,T,mu_s=0.15,mu_d=-0.15,temperature_K=300)
print(f'Toy coherent current at 0.30 eV chemical-potential difference: {I:.4e} A')
print('Reminder: this is a numerical benchmark, not a material/device prediction.')


## 3. Noise and electrothermal utilities

In [ ]:
from noise.psd import one_over_f, lorentzian_rts
from electrothermal.rc import simulate_temperature
f=np.logspace(0,4,300)
psd=one_over_f(f, amplitude=1e-10, exponent=1.0, white=1e-14)+lorentzian_rts(f, delta_i=1e-6, tau=1e-3)
plt.figure(figsize=(7,4)); plt.loglog(f,psd); plt.xlabel('Frequency (Hz)'); plt.ylabel('PSD (A²/Hz)'); plt.title('Synthetic noise components'); plt.grid(True,which='both'); plt.show()
t=np.linspace(0,0.02,1001); power=np.where(t<0.01,1e-3,0.0)
Temp=simulate_temperature(t,power,Tamb_K=300,Rth_K_per_W=1000,Cth_J_per_K=1e-5)
plt.figure(figsize=(7,4)); plt.plot(t,Temp); plt.xlabel('Time (s)'); plt.ylabel('Temperature (K)'); plt.title('Lumped electrothermal transient'); plt.grid(True); plt.show()


## 4. Smoke tests and physics checks

In [ ]:
from negf.chain import surface_gf_1d, transmission_chain
from compact.models import toy_fet_current
from spice.netlist import dc_resistor_divider
# Hermitian real-space chain and bounded transmission for the toy model
assert np.isfinite(surface_gf_1d(0.2,0.0,-1.0))
assert np.all(np.isfinite(T)) and np.all(T >= -1e-9)
assert np.max(T) < 1.1, np.max(T)
# Current should reverse sign if source/drain chemical potentials are swapped
I_forward=landauer_current_ev(E,T,0.1,-0.1,300)
I_reverse=landauer_current_ev(E,T,-0.1,0.1,300)
assert I_forward > 0 and I_reverse < 0
assert np.all(np.asarray(toy_fet_current(np.array([0.0,1.0]),0.1)) >= 0)
assert '.op' in dc_resistor_divider()
print('Smoke tests passed. Note: broader validation remains required before scientific use.')


## 5. Export repository ZIP and download it

In [ ]:
import zipfile
ZIP=Path('/content/quantum2circuit_starter.zip')
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob('*'):
        if p.is_file(): z.write(p, p.relative_to(ROOT.parent))
print('Created:', ZIP)
try:
    from google.colab import files
    files.download(str(ZIP))
except Exception:
    print('Outside Colab, use the notebook environment file browser to retrieve:', ZIP)


## 6. Roadmap for replacing placeholders with research-grade components

1. Add ASE structures and DFT input/output parsers; capture pseudopotential hashes and convergence metadata.
2. Add Wannier90 Hamiltonian import and interpolation-error reports.
3. Replace the toy chain with sparse/device-specific Hamiltonians and robust surface Green-function decimation.
4. Implement self-consistent electrostatics and charge from lesser Green functions; test charge/current conservation.
5. Add uncertainty-aware contact/defect fitting and model comparison on held-out data.
6. Add spin-orbit and valley operators only with validated basis conventions and symmetries.
7. Fit noise models with calibrated instrument floors and uncertainty estimates.
8. Export compact model equations to ngspice and compare circuits against measured device behavior.
9. Keep proprietary/controlled data out of public releases until IP, sponsor, privacy and export-control reviews are complete.
